In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Silver Layer

#####DATA ACCESS USING APP

###DATA Loading



In [0]:
df_cal=spark.read.format('csv').options(header=True, inferSchema=True).load('abfss://raw@adfadvdataset.dfs.core.windows.net/calender')

In [0]:
df_category=spark.read.format('csv').options(header=True, inferSchema=True).load('abfss://raw@adfadvdataset.dfs.core.windows.net/category_lookup')

In [0]:
df_cust_lookup=spark.read.format('csv').options(header=True, inferSchema=True).load('abfss://raw@adfadvdataset.dfs.core.windows.net/customer_lookup')

In [0]:
df_discount=spark.read.format('csv').options(header=True, inferSchema=True).load('abfss://raw@adfadvdataset.dfs.core.windows.net/discount')

In [0]:
df_product_lookup=spark.read.format('csv').options(header=True, inferSchema=True).load('abfss://raw@adfadvdataset.dfs.core.windows.net/product_lookup')

In [0]:
df_product_subcategory_lookup=spark.read.format('csv').options(header=True, inferSchema=True).load('abfss://raw@adfadvdataset.dfs.core.windows.net/product_subcategory_lookup')

In [0]:
df_returns_data=spark.read.format('csv').options(header=True, inferSchema=True).load('abfss://raw@adfadvdataset.dfs.core.windows.net/returns_data')

In [0]:
df_sales_data=spark.read.format('csv').options(header=True, inferSchema=True).load('abfss://raw@adfadvdataset.dfs.core.windows.net/sales*')

In [0]:
df_territory_lookup=spark.read.format('csv').options(header=True, inferSchema=True).load('abfss://raw@adfadvdataset.dfs.core.windows.net/territory_lookup')

###calender

In [0]:
df_cal=df_cal.withColumn('Year',year(col('Date')))\
         .withColumn('Month',month(col('Date')))

In [0]:
df_cal.write.format('parquet')\
    .mode('overwrite')\
    .option("path","abfss://silver@adfadvdataset.dfs.core.windows.net/AdventureWorks_calender")\
    .save()

###customer lookup


In [0]:
df_cust_lookup.display()

In [0]:
df_cust_lookup=df_cust_lookup.withColumn("fullname",concat(col("Prefix"),lit(' '),col("FirstName"),lit(' '),col("LastName")))
df_cust_lookup.display()

In [0]:
df_cust_lookup = df_cust_lookup.withColumn('fullname', concat_ws(' ', col("Prefix"), col("FirstName"), col("LastName")))
df_cust_lookup.display()


In [0]:
df_cust_lookup.write.format('parquet')\
    .mode('overwrite')\
    .option("path","abfss://silver@adfadvdataset.dfs.core.windows.net/AdventureWorks_customer")\
    .save()
df_cust_lookup.display()

###Category

In [0]:
df_category.write.format('parquet')\
    .mode('overwrite')\
    .option("path","abfss://silver@adfadvdataset.dfs.core.windows.net/AdventureWorks_category")\
    .save()
df_category.display()

###product

In [0]:
df_product_lookup.display()

In [0]:
df_product_lookup = df_product_lookup.withColumn('ProductSKU', split(col("ProductSKU"),'-')[0])\
                                    .withColumn('ProductName', split(col("ProductName"),' ')[0])

In [0]:
df_product_lookup.display()

In [0]:
df_product_lookup.write.format('parquet')\
    .mode('overwrite')\
    .option("path","abfss://silver@adfadvdataset.dfs.core.windows.net/AdventureWorks_Product")\
    .save()
df_product_lookup.display()

###RETURNS

In [0]:
df_returns_data.display()

In [0]:
df_returns_data.write.format('parquet')\
    .mode('overwrite')\
    .option("path","abfss://silver@adfadvdataset.dfs.core.windows.net/AdventureWorks_Returns")\
    .save()


###Territory

In [0]:
df_territory_lookup.write.format('parquet')\
    .mode('overwrite')\
    .option("path","abfss://silver@adfadvdataset.dfs.core.windows.net/AdventureWorks_Terrirtory")\
    .save()
df_territory_lookup.display()

###Sales

In [0]:
df_sales_data.display()

In [0]:
df_sales_data=df_sales_data.withColumn('StockDate',to_timestamp('StockDate'))

In [0]:
df_sales_data=df_sales_data.withColumn('OrderNumber', regexp_replace(col('OrderNumber'),'S','T'))

In [0]:
df_sales_data=df_sales_data.withColumn('Multiply',col('OrderLineItem')*col('OrderQuantity'))

In [0]:
df_sales_data.display()

###Sales Analysis

In [0]:
df_sales_data.groupby('OrderDate').agg(count('OrderNumber')).alias('Total_Orders').display()

In [0]:
df_sales_data.write.format('parquet')\
    .mode('overwrite')\
    .option("path","abfss://silver@adfadvdataset.dfs.core.windows.net/AdventureWorks_Sales")\
    .save()